In [1]:
import gradio as gr
from typing import TypedDict, Annotated, List
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
import os

class PlannerState(TypedDict):
    messages: Annotated[List[HumanMessage | AIMessage], "The messages in the conversation"]
    city: str
    interests: List[str]
    itinerary: str

# Define the LLM - using environment variable for security
llm = ChatGroq(
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY", ""),
    model_name="llama-3.3-70b-versatile"
)

# Define the itinerary prompt
itinerary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Create a detailed day trip itinerary for {city} based on the user's interests: {interests}. Provide a well-structured itinerary with specific recommendations for places to visit, activities, and timing."),
    ("human", "Create an itinerary for my day trip."),
])

def input_city(city: str, state: PlannerState) -> PlannerState:
    return {
        **state,
        "city": city,
        "messages": state['messages'] + [HumanMessage(content=city)],
    }

def input_interests(interests: str, state: PlannerState) -> PlannerState:
    return {
        **state,
        "interests": [interest.strip() for interest in interests.split(',')],
        "messages": state['messages'] + [HumanMessage(content=interests)],
    }

def create_itinerary(state: PlannerState) -> str:
    response = llm.invoke(itinerary_prompt.format_messages(city=state['city'], interests=", ".join(state['interests'])))
    state["itinerary"] = response.content
    state["messages"] += [AIMessage(content=response.content)]
    return response.content

# Define the Gradio application
def travel_planner(city: str, interests: str):
    if not city or not interests:
        return "⚠️ Please enter both a city and your interests!"
    
    # Initialize state
    state = {
        "messages": [],
        "city": "",
        "interests": [],
        "itinerary": "",
    }

    # Process the city and interests inputs
    state = input_city(city, state)
    state = input_interests(interests, state)

    # Generate the itinerary
    try:
        itinerary = create_itinerary(state)
        return itinerary
    except Exception as e:
        return f"❌ Error generating itinerary: {str(e)}"

# Build the Gradio interface with custom styling
with gr.Blocks(theme=gr.themes.Soft()) as interface:
    gr.Markdown(
        """
        # 🌍 Trip Craft - AI Travel Planner
        ### Plan your perfect day trip with AI-powered recommendations
        Enter your destination city and interests to get a personalized itinerary!
        """
    )
    
    with gr.Row():
        with gr.Column():
            city_input = gr.Textbox(
                label="🏙️ City",
                placeholder="e.g., Paris, Tokyo, New York",
                lines=1
            )
            interests_input = gr.Textbox(
                label="✨ Your Interests",
                placeholder="e.g., museums, food, architecture, shopping",
                lines=2
            )
            submit_btn = gr.Button("Generate Itinerary", variant="primary", size="lg")
        
        with gr.Column():
            output = gr.Textbox(
                label="📋 Your Personalized Itinerary",
                lines=15,
                max_lines=20
            )
    
    gr.Examples(
        examples=[
            ["Paris", "art, food, romance"],
            ["Tokyo", "technology, anime, temples"],
            ["New York", "broadway, museums, shopping"],
            ["Rome", "history, architecture, food"],
        ],
        inputs=[city_input, interests_input],
    )
    
    submit_btn.click(
        fn=travel_planner,
        inputs=[city_input, interests_input],
        outputs=output
    )

# Launch the Gradio application
if __name__ == "__main__":
    interface.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [2]:
import gradio as gr
import re
import json
from urllib.parse import quote_plus
from typing import TypedDict, Annotated, List
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
import os

class PlannerState(TypedDict):
    messages: Annotated[List[HumanMessage | AIMessage], "The messages in the conversation"]
    city: str
    interests: List[str]
    itinerary: str

# NOTE: llama-3.3-70b-versatile was deprecated by Groq (announced June 17, 2026).
# Consider switching model_name to "openai/gpt-oss-120b" or "qwen/qwen3.6-27b"
# if you start seeing a model_decommissioned error.
llm = ChatGroq(
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY", ""),
    model_name="llama-3.3-70b-versatile"
)

# ---------------------------------------------------------------------------
# Prompt asks for structured JSON (day/time/activity/place/description)
# instead of free-form prose, so we can reliably build Maps links and
# show "why this place is famous" without fragile text parsing.
# ---------------------------------------------------------------------------
itinerary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful travel assistant. Create a detailed day trip itinerary "
     "for {city} based on the user's interests: {interests}.\n\n"
     "Return ONLY a JSON array (no markdown fences, no extra commentary, no "
     "leading/trailing text). Each element must be an object with exactly "
     "these keys:\n"
     '  - "day": integer day number (use 1 for a single-day trip)\n'
     '  - "time": string, e.g. "09:00 AM"\n'
     '  - "activity": short description, e.g. "Visit Mysore Palace"\n'
     '  - "place": just the venue/place name to search on a map, e.g. '
     '"Mysore Palace" (no verbs like "Visit", no times)\n'
     '  - "description": 1-2 sentences explaining what this place/venue is '
     'known or famous for (historical significance, specialty dish, view, '
     'architecture, etc.). Keep it factual and concise — no fluff.\n\n'
     "Order the array chronologically. Include 5-8 stops (breakfast, "
     "sightseeing, lunch, more activities, dinner) where appropriate."),
    ("human", "Create an itinerary for my day trip."),
])


def input_city(city: str, state: PlannerState) -> PlannerState:
    return {
        **state,
        "city": city,
        "messages": state['messages'] + [HumanMessage(content=city)],
    }


def input_interests(interests: str, state: PlannerState) -> PlannerState:
    return {
        **state,
        "interests": [interest.strip() for interest in interests.split(',')],
        "messages": state['messages'] + [HumanMessage(content=interests)],
    }


def generate_maps_link(place_name: str, city: str = "") -> str:
    """
    Build a Google Maps 'search' URL for a given place.

    Uses urllib.parse.quote_plus() rather than a manual .replace(" ", "+")
    because it also safely encodes commas, ampersands, apostrophes, etc.
    that commonly appear in place names, while still turning spaces
    into '+' as required.

    Appends the city name when it isn't already part of the place name,
    to disambiguate generic names (e.g. "Chamundi Hill" -> "Chamundi Hill Mysore").
    """
    place_name = (place_name or "").strip()
    if not place_name:
        return ""

    if city and city.lower() not in place_name.lower():
        query = f"{place_name} {city}"
    else:
        query = place_name

    encoded_query = quote_plus(query)
    return f"https://www.google.com/maps/search/?api=1&query={encoded_query}"


def parse_structured_itinerary(raw_response: str) -> list:
    """
    Safely parse the LLM's JSON response into a Python list.
    Strips ```json fences if the model adds them despite instructions,
    and falls back to an empty list instead of crashing on bad JSON.
    """
    cleaned = raw_response.strip()
    cleaned = re.sub(r"^```(json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()

    try:
        stops = json.loads(cleaned)
        if isinstance(stops, list):
            return stops
    except json.JSONDecodeError:
        pass
    return []


def format_itinerary_html(stops: list, city: str) -> str:
    """
    Render the structured stops as HTML, including:
      - time + activity
      - a short "why famous" description
      - a clickable Google Maps link (opens in a new tab)
    """
    if not stops:
        return "<p>⚠️ Could not parse the itinerary. Please try generating it again.</p>"

    html_parts = []
    current_day = None

    for stop in stops:
        day = stop.get("day", 1)
        time = stop.get("time", "")
        activity = stop.get("activity", "")
        place = stop.get("place") or activity
        description = stop.get("description", "").strip()

        if day != current_day:
            if current_day is not None:
                html_parts.append("<hr>")
            html_parts.append(f"<h3>Day {day}</h3>")
            current_day = day

        maps_url = generate_maps_link(place, city)

        # Only render the description block if the model actually returned one
        description_html = (
            f'<p style="margin: 0 0 4px 0; color: #555; font-size: 0.92em;">ℹ️ {description}</p>'
            if description else ""
        )

        html_parts.append(f"""
        <div style="margin-bottom: 14px;">
            <p style="font-weight: bold; margin: 0 0 2px 0;">{time}</p>
            <p style="margin: 0 0 4px 0;">{activity}</p>
            {description_html}
            <p style="margin: 0;">
                📍 <a href="{maps_url}" target="_blank" rel="noopener noreferrer">
                    Open in Google Maps
                </a>
            </p>
        </div>
        <hr style="border: none; border-top: 1px dashed #ccc; margin: 8px 0;">
        """)

    return "\n".join(html_parts)


def create_itinerary(state: PlannerState) -> str:
    response = llm.invoke(
        itinerary_prompt.format_messages(city=state['city'], interests=", ".join(state['interests']))
    )
    stops = parse_structured_itinerary(response.content)
    formatted_html = format_itinerary_html(stops, state['city'])

    state["itinerary"] = formatted_html
    state["messages"] += [AIMessage(content=response.content)]
    return formatted_html


def travel_planner(city: str, interests: str):
    if not city or not interests:
        return "<p>⚠️ Please enter both a city and your interests!</p>"

    state = {
        "messages": [],
        "city": "",
        "interests": [],
        "itinerary": "",
    }
    state = input_city(city, state)
    state = input_interests(interests, state)

    try:
        itinerary = create_itinerary(state)
        return itinerary
    except Exception as e:
        return f"<p>❌ Error generating itinerary: {str(e)}</p>"


with gr.Blocks(theme=gr.themes.Soft()) as interface:
    gr.Markdown(
        """
        # 🌍 Trip Craft - AI Travel Planner
        ### Plan your perfect day trip with AI-powered recommendations
        Enter your destination city and interests to get a personalized itinerary!
        """
    )

    with gr.Row():
        with gr.Column():
            city_input = gr.Textbox(label="🏙️ City", placeholder="e.g., Paris, Tokyo, New York", lines=1)
            interests_input = gr.Textbox(label="✨ Your Interests", placeholder="e.g., museums, food, architecture, shopping", lines=2)
            submit_btn = gr.Button("Generate Itinerary", variant="primary", size="lg")

        with gr.Column():
            # gr.HTML (not gr.Textbox) so the <a target="_blank"> Maps links
            # actually render as clickable links instead of plain text
            output = gr.HTML(label="📋 Your Personalized Itinerary")

    gr.Examples(
        examples=[
            ["Paris", "art, food, romance"],
            ["Tokyo", "technology, anime, temples"],
            ["New York", "broadway, museums, shopping"],
            ["Rome", "history, architecture, food"],
        ],
        inputs=[city_input, interests_input],
    )

    submit_btn.click(fn=travel_planner, inputs=[city_input, interests_input], outputs=output)

if __name__ == "__main__":
    interface.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
